In [1]:
!pip install spacy

In [2]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 1.8 MB/s eta 0:00:07
     -- ------------------------------------- 0.8/12.8 MB 1.4 MB/s eta 0:00:09
     ---- ----------------------------------- 1.3/12.8 MB 1.6 MB/s eta 0:00:07
     ----- ---------------------------------- 1.8/12.8 MB 1.8 MB/s eta 0:00:07
     ------ --------------------------------- 2.1/12.8 MB 1.9 MB/s eta 0:00:06
     -------- ------------------------------- 2.6/12.8 MB 2.0 MB/s eta 0:00:06
     ---------- ----------------------------- 3.4/12.8 MB 2.1 MB/s eta 0:00:05
     ------------ --------------------------- 3.9/12.8 MB 2.2 MB/s eta 0:00:04
     ------------- -------------------------- 4.5/12.8 MB 2.3 MB/s eta 0:00:04
     --------------- ------------------------ 5.0/12.8 MB 2.3 MB/s eta 0:

In [3]:
import pandas as pd
import spacy

In [4]:
df = pd.read_csv("clean_jobs.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10100, 12)


,job_id,job_title,company,location,job_description,experience,education,salary,job_type,clean_description,original_length,clean_length
0,JOB07402,Frontend Developer,DataBridge Analytics,"Gurgaon, India",We are looking for a Frontend Developer to joi...,1-3 years,B.E.,9-14 LPA,Full-time,we are looking for a frontend developer to joi...,362,351
1,JOB05835,Software Engineer,Vertex Digital,"Mumbai, India",We are looking for a Software Engineer to join...,3-5 years,Bachelor's Degree,12-16 LPA,Internship,we are looking for a software engineer to join...,342,333
2,JOB02123,Machine Learning Engineer,CloudSphere,"Mumbai, India",We are looking for a Machine Learning Engineer...,1-3 years,B.E.,8-11 LPA,Full-time,we are looking for a machine learning engineer...,364,354
3,JOB08789,Cloud Engineer,Apex Solutions,"Mumbai, India",We are looking for a Cloud Engineer to join ou...,3-5 years,Master's Degree,11-16 LPA,Full-time,we are looking for a cloud engineer to join ou...,361,350
4,JOB00305,Frontend Developer,Quantix Technologies,"Noida, India",We are looking for a Frontend Developer to joi...,3-5 years,B.Tech,10-12 LPA,Full-time,we are looking for a frontend developer to joi...,366,355


In [5]:
nlp = spacy.load("en_core_web_sm")

In [6]:
text = df["clean_description"].iloc[0]

doc = nlp(text)

In [7]:
for entity in doc.ents:
    print(entity.text, "->", entity.label_)

git html typescript -> PERSON


In [8]:
custom_entities = {

    "Python": "SKILL",
    "SQL": "SKILL",
    "Java": "SKILL",

    "Power BI": "TOOL",
    "Tableau": "TOOL",
    "Excel": "TOOL",

    "TensorFlow": "TECHNOLOGY",
    "PyTorch": "TECHNOLOGY",
    "Docker": "TECHNOLOGY",

    "PostgreSQL": "DATABASE",
    "MySQL": "DATABASE",
    "MongoDB": "DATABASE",

    "AWS": "CLOUD_PLATFORM",
    "Azure": "CLOUD_PLATFORM",
    "GCP": "CLOUD_PLATFORM"
}

In [9]:
def extract_custom_entities(text):

    found_entities = []

    text = str(text).lower()

    for entity, label in custom_entities.items():

        if entity.lower() in text:

            found_entities.append(
                (entity, label)
            )

    return found_entities

In [10]:
df["custom_entities"] = df[
    "clean_description"
].apply(extract_custom_entities)

In [12]:
df[[
    "job_title",
    "custom_entities"
]].head(10)

,job_title,custom_entities
0,Frontend Developer,"[(Java, SKILL)]"
1,Software Engineer,"[(Python, SKILL), (SQL, SKILL)]"
2,Machine Learning Engineer,"[(Python, SKILL), (TensorFlow, TECHNOLOGY), (P..."
3,Cloud Engineer,"[(Docker, TECHNOLOGY), (AWS, CLOUD_PLATFORM), ..."
4,Frontend Developer,"[(Java, SKILL)]"
5,Data Analyst,"[(Python, SKILL), (SQL, SKILL), (Power BI, TOO..."
6,Machine Learning Engineer,"[(Python, SKILL), (SQL, SKILL), (TensorFlow, T..."
7,DevOps Engineer,"[(AWS, CLOUD_PLATFORM)]"
8,QA Engineer,"[(Python, SKILL), (SQL, SKILL)]"
9,Business Analyst,"[(SQL, SKILL), (Power BI, TOOL), (Tableau, TOO..."


In [15]:
for index, row in df.head(10).iterrows():

    print("Job:", row["job_title"])

    for entity, label in row["custom_entities"]:
        print("  ", entity, "->", label)

    print()

Job: Frontend Developer
   Java -> SKILL

Job: Software Engineer
   Python -> SKILL
   SQL -> SKILL

Job: Machine Learning Engineer
   Python -> SKILL
   TensorFlow -> TECHNOLOGY
   PyTorch -> TECHNOLOGY
   Docker -> TECHNOLOGY
   AWS -> CLOUD_PLATFORM

Job: Cloud Engineer
   Docker -> TECHNOLOGY
   AWS -> CLOUD_PLATFORM
   Azure -> CLOUD_PLATFORM

Job: Frontend Developer
   Java -> SKILL

Job: Data Analyst
   Python -> SKILL
   SQL -> SKILL
   Power BI -> TOOL
   Tableau -> TOOL
   Excel -> TOOL

Job: Machine Learning Engineer
   Python -> SKILL
   SQL -> SKILL
   TensorFlow -> TECHNOLOGY
   PyTorch -> TECHNOLOGY

Job: DevOps Engineer
   AWS -> CLOUD_PLATFORM

Job: QA Engineer
   Python -> SKILL
   SQL -> SKILL

Job: Business Analyst
   SQL -> SKILL
   Power BI -> TOOL
   Tableau -> TOOL
   Excel -> TOOL



In [16]:
from collections import Counter

category_counts = Counter()

for entity_list in df["custom_entities"]:

    for entity, label in entity_list:
        category_counts[label] += 1

print(category_counts)

Counter({'SKILL': 13466, 'TECHNOLOGY': 4999, 'TOOL': 4830, 'CLOUD_PLATFORM': 3587, 'DATABASE': 730})


In [17]:
df.to_csv(
    "custom_ner_results.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!
